# Log-rank Test para Colapso de Categorias

**Objetivo:** usar o log-rank test para identificar quais níveis de uma variável categórica
têm curvas de sobrevivência estatisticamente iguais — e portanto podem ser colapsados.

**Caso de uso:** variável `uf` com 10 estados, onde alguns têm perfis de risco similares
e outros claramente diferentes. O dataset é construído artificialmente para que você
saiba de antemão qual é a resposta certa e possa validar se a função encontra.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import unittest
import io
from itertools import combinations
from scipy.stats import chi2
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from scipy.spatial.distance import squareform
from typing import Dict, List, Optional, Tuple

from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test, multivariate_logrank_test

plt.style.use('seaborn-v0_8-whitegrid')
PALETTE = ['#2196F3','#E91E63','#4CAF50','#FF9800','#9C27B0',
           '#00BCD4','#FF5722','#607D8B','#8BC34A','#FFC107']
np.random.seed(42)
print('✅ Setup OK')

## 1. Dataset Sintético — UFs com grupos de risco conhecidos

Construímos o dataset com **três grupos de risco conhecidos** e tamanhos de amostra
intencionalmente diferentes por UF.
Isso nos permite validar se a função recupera a estrutura verdadeira.

```
Grupo BAIXO risco  → SP, MG, RS        (escala Weibull alta → sobrevive mais)
Grupo MÉDIO risco  → RJ, PR, SC        (escala intermediária)
Grupo ALTO risco   → BA, CE, PE, AM    (escala baixa → default mais rápido)

Tamanhos propositalmente desiguais:
  SP = 800 contratos  (grande, curva estável)
  AM =  40 contratos  (pequeno, curva instável)
```

A função deve:
- Detectar que SP/MG/RS são indistinguíveis entre si
- Detectar que BA/CE/PE/AM são indistinguíveis entre si  
- Detectar que os três grupos são distintos entre si
- Sinalizar incerteza alta para AM (n pequeno)

In [ ]:
# ── Gabarito: estrutura verdadeira que a função deve recuperar ────────────────
TRUE_GROUPS = {
    'baixo_risco' : {'ufs': ['SP', 'MG', 'RS'], 'scale': 25.0},
    'medio_risco' : {'ufs': ['RJ', 'PR', 'SC'], 'scale': 15.0},
    'alto_risco'  : {'ufs': ['BA', 'CE', 'PE', 'AM'], 'scale': 8.0},
}

# Tamanhos propositalmente desiguais — AM é pequeno para testar incerteza
UF_SIZES = {
    'SP': 800, 'MG': 500, 'RS': 300,   # baixo risco — bem amostrados
    'RJ': 400, 'PR': 250, 'SC': 150,   # médio risco
    'BA': 200, 'CE': 180, 'PE': 120,   # alto risco
    'AM':  40,                          # alto risco — pequeno → incerteza alta
}


def make_uf_dataset(
    true_groups: Dict,
    uf_sizes: Dict,
    shape: float = 1.2,
    max_followup: int = 48,
    random_state: int = 42
) -> pd.DataFrame:
    """
    Gera dataset de contratos com UF e tempos de evento.

    O tempo até o evento segue distribuição Weibull com escala
    definida pelo grupo de risco da UF. Contratos além do
    follow-up máximo são censurados.

    Parameters
    ----------
    true_groups  : dict com grupos, UFs e escala Weibull de cada grupo
    uf_sizes     : dict com número de contratos por UF
    shape        : parâmetro de forma Weibull (> 1 → hazard crescente)
    max_followup : MOB máximo de observação (censura administrativa)
    random_state : semente

    Returns
    -------
    DataFrame com colunas: id, uf, grupo_verdadeiro, time, event
    """
    rng = np.random.default_rng(random_state)
    rows = []

    # Mapeia UF → escala e grupo
    uf_to_scale = {}
    uf_to_group = {}
    for group_name, info in true_groups.items():
        for uf in info['ufs']:
            uf_to_scale[uf] = info['scale']
            uf_to_group[uf] = group_name

    for uf, n in uf_sizes.items():
        scale = uf_to_scale[uf]

        # Adiciona ruído por UF para não serem idênticas dentro do grupo
        # (mais realista — UFs do mesmo grupo têm risco similar, não igual)
        uf_noise = rng.uniform(0.85, 1.15)
        t_event  = rng.weibull(shape, n) * scale * uf_noise
        t_cens   = rng.uniform(max_followup * 0.5, max_followup, n)

        t_obs  = np.minimum(t_event, t_cens)
        t_obs  = np.clip(np.round(t_obs).astype(int), 1, max_followup)
        evento = (t_event <= t_cens).astype(int)

        for i in range(n):
            rows.append({
                'id'              : f'{uf}_{i:04d}',
                'uf'              : uf,
                'grupo_verdadeiro': uf_to_group[uf],
                'time'            : t_obs[i],
                'event'           : evento[i],
            })

    df = pd.DataFrame(rows).sample(frac=1, random_state=random_state).reset_index(drop=True)
    return df


df = make_uf_dataset(TRUE_GROUPS, UF_SIZES)

print('Shape:', df.shape)
print('\nEstatísticas por UF:')
stats = df.groupby(['uf', 'grupo_verdadeiro']).agg(
    n_contratos=('id', 'count'),
    n_eventos=('event', 'sum'),
    event_rate=('event', 'mean'),
    median_time=('time', 'median'),
).round(3)
print(stats.to_string())

## 2. Funções Principais

In [ ]:
def pairwise_logrank(
    df: pd.DataFrame,
    category_col: str,
    time_col: str,
    event_col: str,
    min_events: int = 10
) -> pd.DataFrame:
    """
    Calcula o log-rank test para todos os pares de níveis de uma
    variável categórica.

    Para cada par (A, B), a hipótese nula é:
      H₀: a curva de sobrevivência de A é igual à de B
      H₁: as curvas são diferentes

    p-value alto (> 0.05) → não rejeitamos H₀ → curvas similares
                          → candidatos a colapso
    p-value baixo         → curvas diferentes → manter separados

    Considera incerteza pelo tamanho amostral:
    pares com poucos eventos são flagged como 'low_power' —
    p-value alto pode ser por falta de poder, não por similaridade real.

    Parameters
    ----------
    df            : DataFrame
    category_col  : coluna categórica (ex: 'uf')
    time_col      : coluna de tempo
    event_col     : coluna de evento (binário)
    min_events    : mínimo de eventos por nível para ter poder estatístico

    Returns
    -------
    DataFrame com colunas:
      level_a, level_b       : par testado
      n_a, n_b               : tamanho de cada grupo
      events_a, events_b     : número de eventos em cada grupo
      logrank_stat           : estatística do teste
      pvalue                 : p-value (bilateral)
      similar                : True se p > 0.05 (não rejeita H₀)
      low_power              : True se algum grupo tem < min_events eventos
      interpretation         : texto legível da conclusão
    """
    levels = sorted(df[category_col].dropna().unique())
    rows   = []

    for level_a, level_b in combinations(levels, 2):
        mask_a = df[category_col] == level_a
        mask_b = df[category_col] == level_b

        t_a = df.loc[mask_a, time_col]
        t_b = df.loc[mask_b, time_col]
        e_a = df.loc[mask_a, event_col]
        e_b = df.loc[mask_b, event_col]

        n_a, n_b         = mask_a.sum(), mask_b.sum()
        events_a         = int(e_a.sum())
        events_b         = int(e_b.sum())
        low_power        = (events_a < min_events) or (events_b < min_events)

        try:
            result = logrank_test(t_a, t_b,
                                  event_observed_A=e_a,
                                  event_observed_B=e_b)
            stat  = round(result.test_statistic, 4)
            pval  = round(result.p_value, 6)
        except Exception:
            stat, pval = None, None

        similar = (pval is not None) and (pval > 0.05)

        if pval is None:
            interp = 'Erro no cálculo'
        elif low_power:
            interp = ('⚠️  p-value alto mas PODER BAIXO — '
                      'similaridade incerta por n pequeno'
                      if similar else
                      '⚠️  Diferença detectada mas poder baixo — resultado frágil')
        elif similar:
            interp = '✅ Curvas similares — candidatos a colapso'
        else:
            interp = '❌ Curvas diferentes — manter separados'

        rows.append({
            'level_a'       : level_a,
            'level_b'       : level_b,
            'n_a'           : n_a,
            'n_b'           : n_b,
            'events_a'      : events_a,
            'events_b'      : events_b,
            'logrank_stat'  : stat,
            'pvalue'        : pval,
            'similar'       : similar,
            'low_power'     : low_power,
            'interpretation': interp,
        })

    return pd.DataFrame(rows).sort_values('pvalue', ascending=False)


def suggest_collapsing(
    pairwise_df: pd.DataFrame,
    pvalue_threshold: float = 0.05,
    ignore_low_power: bool = False
) -> List[List[str]]:
    """
    A partir da matriz de p-values pairwise, sugere grupos de níveis
    que podem ser colapsados usando clustering hierárquico.

    Lógica:
      - Transforma p-values em uma distância: dist = -log(p)
        → p alto (similar) = distância pequena
        → p baixo (diferente) = distância grande
      - Aplica clustering hierárquico (linkage completo)
      - Corta o dendrograma no threshold definido

    Parameters
    ----------
    pairwise_df       : saída de pairwise_logrank()
    pvalue_threshold  : threshold para considerar curvas similares
    ignore_low_power  : se True, inclui pares com low_power na similaridade

    Returns
    -------
    Lista de listas, onde cada lista interna é um grupo de níveis
    que podem ser colapsados
    """
    levels = sorted(set(
        pairwise_df['level_a'].tolist() + pairwise_df['level_b'].tolist()
    ))
    n = len(levels)
    idx = {lv: i for i, lv in enumerate(levels)}

    # Matriz de distância: -log(p+ε)
    # p alto → log(p) próximo de 0 → distância pequena → similar
    dist_matrix = np.full((n, n), 50.0)  # default: muito distante
    np.fill_diagonal(dist_matrix, 0.0)

    for _, row in pairwise_df.iterrows():
        if row['pvalue'] is None:
            continue
        if row['low_power'] and not ignore_low_power:
            # Low power: não consideramos como similar com segurança
            dist = 50.0
        else:
            dist = -np.log(row['pvalue'] + 1e-10)

        i, j = idx[row['level_a']], idx[row['level_b']]
        dist_matrix[i, j] = dist
        dist_matrix[j, i] = dist

    # Clustering hierárquico
    condensed = squareform(dist_matrix)
    Z         = linkage(condensed, method='complete')

    # Threshold de corte: -log(pvalue_threshold)
    cut = -np.log(pvalue_threshold)
    labels = fcluster(Z, t=cut, criterion='distance')

    groups = {}
    for level, cluster_id in zip(levels, labels):
        groups.setdefault(int(cluster_id), []).append(level)

    return list(groups.values())


def plot_pairwise_pvalues(
    pairwise_df: pd.DataFrame,
    pvalue_threshold: float = 0.05
):
    """
    Heatmap dos p-values do log-rank pairwise.

    Verde = p alto = curvas similares = candidato a colapso
    Vermelho = p baixo = curvas diferentes = manter separados

    Células com ⚠️ indicam low_power (n pequeno).

    Parameters
    ----------
    pairwise_df       : saída de pairwise_logrank()
    pvalue_threshold  : linha de corte visual
    """
    levels = sorted(set(
        pairwise_df['level_a'].tolist() + pairwise_df['level_b'].tolist()
    ))
    n   = len(levels)
    idx = {lv: i for i, lv in enumerate(levels)}

    pval_matrix     = np.full((n, n), np.nan)
    lowpower_matrix = np.zeros((n, n), dtype=bool)
    np.fill_diagonal(pval_matrix, 1.0)

    for _, row in pairwise_df.iterrows():
        if row['pvalue'] is None:
            continue
        i, j = idx[row['level_a']], idx[row['level_b']]
        pval_matrix[i, j] = row['pvalue']
        pval_matrix[j, i] = row['pvalue']
        lowpower_matrix[i, j] = row['low_power']
        lowpower_matrix[j, i] = row['low_power']

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # ── Heatmap de p-values ───────────────────────────────────────────────────
    ax = axes[0]
    im = ax.imshow(pval_matrix, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
    plt.colorbar(im, ax=ax, label='p-value')

    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xticklabels(levels, rotation=45, ha='right')
    ax.set_yticklabels(levels)

    for i in range(n):
        for j in range(n):
            if np.isnan(pval_matrix[i, j]):
                continue
            text = f'{pval_matrix[i, j]:.2f}'
            if lowpower_matrix[i, j]:
                text += '\n⚠️'
            color = 'black' if 0.2 < pval_matrix[i, j] < 0.8 else 'white'
            ax.text(j, i, text, ha='center', va='center',
                    fontsize=7.5, color=color, fontweight='bold')

    ax.set_title(
        'P-values Log-rank Pairwise\n'
        'Verde = similar (candidato a colapso) | Vermelho = diferente\n'
        '⚠️ = low power (n pequeno — similaridade incerta)',
        fontweight='bold'
    )

    # ── Dendrograma ───────────────────────────────────────────────────────────
    ax2  = axes[1]
    dist = -np.log(np.where(pval_matrix == 0, 1e-10, pval_matrix))
    np.fill_diagonal(dist, 0)

    try:
        Z = linkage(squareform(dist), method='complete')
        dendrogram(
            Z, labels=levels, ax=ax2,
            color_threshold=-np.log(pvalue_threshold),
            above_threshold_color='gray'
        )
        ax2.axhline(
            -np.log(pvalue_threshold),
            color='red', linestyle='--', linewidth=1.5,
            label=f'Threshold p={pvalue_threshold}'
        )
        ax2.set_title(
            'Dendrograma de Similaridade\n'
            'Grupos abaixo da linha vermelha → candidatos a colapso',
            fontweight='bold'
        )
        ax2.set_ylabel('-log(p-value)  [distância]')
        ax2.tick_params(axis='x', rotation=45)
        ax2.legend()
    except Exception as e:
        ax2.text(0.5, 0.5, f'Dendrograma indisponível:\n{e}',
                 ha='center', va='center', transform=ax2.transAxes)

    plt.tight_layout()
    plt.show()


def plot_km_by_level(
    df: pd.DataFrame,
    category_col: str,
    time_col: str,
    event_col: str,
    suggested_groups: Optional[List[List[str]]] = None
):
    """
    Plota curvas KM por nível da categórica.
    Níveis do mesmo grupo sugerido usam a mesma cor.
    Tamanho da banda de confiança reflete incerteza por n.

    Parameters
    ----------
    df               : DataFrame
    category_col     : coluna categórica
    time_col         : coluna de tempo
    event_col        : coluna de evento
    suggested_groups : saída de suggest_collapsing()
    """
    levels = sorted(df[category_col].dropna().unique())

    # Mapeia nível → cor do grupo sugerido
    color_map = {}
    if suggested_groups:
        for g_idx, group in enumerate(suggested_groups):
            for lv in group:
                color_map[lv] = PALETTE[g_idx % len(PALETTE)]
    else:
        color_map = {lv: PALETTE[i % len(PALETTE)] for i, lv in enumerate(levels)}

    fig, ax = plt.subplots(figsize=(12, 6))

    for lv in levels:
        mask = df[category_col] == lv
        n    = mask.sum()
        evts = df.loc[mask, event_col].sum()

        kmf = KaplanMeierFitter(label=f'{lv} (n={n}, ev={evts})')
        kmf.fit(df.loc[mask, time_col], event_observed=df.loc[mask, event_col])

        # Banda de confiança larga quando n é pequeno — reflete incerteza
        kmf.plot_survival_function(
            ax=ax,
            color=color_map.get(lv, 'black'),
            ci_show=True,
            linewidth=2
        )

    # Legenda de grupos
    if suggested_groups:
        for g_idx, group in enumerate(suggested_groups):
            ax.plot([], [], color=PALETTE[g_idx % len(PALETTE)],
                    linewidth=4, alpha=0.4,
                    label=f'Grupo sugerido {g_idx+1}: {"+".join(sorted(group))}')

    ax.set_title(
        f'Curvas KM por {category_col.upper()}\n'
        'Mesma cor = grupo sugerido para colapso | '
        'Banda larga = mais incerteza (n pequeno)',
        fontweight='bold'
    )
    ax.set_xlabel('Tempo')
    ax.set_ylabel('Sobrevivência')
    ax.legend(fontsize=7, loc='upper right')
    plt.tight_layout()
    plt.show()


def collapsing_report(
    df: pd.DataFrame,
    category_col: str,
    time_col: str,
    event_col: str,
    pvalue_threshold: float = 0.05,
    min_events: int = 10,
    verbose: bool = True
) -> Dict:
    """
    Pipeline completo: roda pairwise log-rank, sugere grupos e
    plota os resultados.

    Parameters
    ----------
    df                : DataFrame
    category_col      : coluna categórica
    time_col          : coluna de tempo
    event_col         : coluna de evento
    pvalue_threshold  : limiar de similaridade
    min_events        : mínimo de eventos por nível (low_power flag)
    verbose           : imprime relatório textual

    Returns
    -------
    dict com:
      'pairwise'         : DataFrame pairwise completo
      'suggested_groups' : lista de grupos sugeridos
      'collapse_map'     : dict nível → rótulo do grupo colapsado
    """
    pairwise = pairwise_logrank(
        df, category_col, time_col, event_col, min_events=min_events
    )
    groups   = suggest_collapsing(pairwise, pvalue_threshold=pvalue_threshold)

    # Mapa de colapso: nível → nome do grupo
    collapse_map = {}
    for g_idx, group in enumerate(groups):
        label = f'grupo_{g_idx+1}__' + '_'.join(sorted(group))
        for lv in group:
            collapse_map[lv] = label

    if verbose:
        print('══════════════════════════════════════════════')
        print(f'  RELATÓRIO DE COLAPSO — {category_col.upper()}')
        print('══════════════════════════════════════════════')
        print(f'  Níveis originais  : {sorted(df[category_col].unique())}')
        print(f'  Grupos sugeridos  : {len(groups)}')
        print()
        for g_idx, group in enumerate(groups):
            n_total  = df[df[category_col].isin(group)].shape[0]
            n_events = df.loc[df[category_col].isin(group), event_col].sum()
            low_pwr  = any(
                pairwise[
                    ((pairwise['level_a'].isin(group)) |
                     (pairwise['level_b'].isin(group)))
                ]['low_power']
            )
            flag = ' ⚠️  (contém UF com low power)' if low_pwr else ''
            print(f'  Grupo {g_idx+1}: {", ".join(sorted(group))}')
            print(f'          n={n_total} | eventos={n_events}{flag}')

        print()
        similar_pairs = pairwise[pairwise['similar'] & ~pairwise['low_power']]
        diff_pairs    = pairwise[~pairwise['similar']]
        lp_pairs      = pairwise[pairwise['low_power']]
        print(f'  Pares similares (p>{pvalue_threshold}, sem low power): {len(similar_pairs)}')
        print(f'  Pares diferentes                                     : {len(diff_pairs)}')
        print(f'  Pares com low power (incerteza)                      : {len(lp_pairs)}')
        print('══════════════════════════════════════════════')

    plot_pairwise_pvalues(pairwise, pvalue_threshold)
    plot_km_by_level(df, category_col, time_col, event_col, groups)

    return {
        'pairwise'        : pairwise,
        'suggested_groups': groups,
        'collapse_map'    : collapse_map,
    }


print('✅ Funções definidas')

## 3. Aplicação no Dataset Sintético

In [ ]:
result = collapsing_report(
    df,
    category_col='uf',
    time_col='time',
    event_col='event',
    pvalue_threshold=0.05,
    min_events=10
)

print('\nPairwise completo (ordenado por p-value):')
print(result['pairwise'][['level_a','level_b','n_a','n_b',
                           'events_a','events_b','pvalue',
                           'similar','low_power']].to_string(index=False))

## 4. Testes Unitários

In [ ]:
class TestPairwiseLogrank(unittest.TestCase):
    """Testes para pairwise_logrank."""

    @classmethod
    def setUpClass(cls):
        cls.df = make_uf_dataset(TRUE_GROUPS, UF_SIZES, random_state=42)

    def test_number_of_pairs(self):
        """Deve retornar C(n_levels, 2) pares."""
        n_levels = self.df['uf'].nunique()
        result   = pairwise_logrank(self.df, 'uf', 'time', 'event')
        expected = n_levels * (n_levels - 1) // 2
        self.assertEqual(len(result), expected)

    def test_pvalue_range(self):
        """P-values devem estar entre 0 e 1."""
        result = pairwise_logrank(self.df, 'uf', 'time', 'event')
        valid  = result['pvalue'].dropna()
        self.assertTrue((valid >= 0).all())
        self.assertTrue((valid <= 1).all())

    def test_same_group_high_pvalue(self):
        """
        Pares dentro do mesmo grupo verdadeiro devem ter p-value alto
        (curvas similares → não rejeita H₀).
        """
        result = pairwise_logrank(self.df, 'uf', 'time', 'event')

        # Pares dentro do grupo baixo_risco (SP, MG, RS) — bem amostrados
        intra_baixo = result[
            result['level_a'].isin(['SP','MG','RS']) &
            result['level_b'].isin(['SP','MG','RS'])
        ]
        # Todos devem ter p > 0.05 (não rejeitar similaridade)
        self.assertTrue(
            (intra_baixo['pvalue'] > 0.05).all(),
            msg=f'Pares intra-grupo baixo risco:\n{intra_baixo[["level_a","level_b","pvalue"]]}'
        )

    def test_different_groups_low_pvalue(self):
        """
        Pares entre grupos diferentes devem ter p-value baixo
        (curvas diferentes → rejeita H₀).
        Foca em pares bem amostrados para evitar low_power.
        """
        result = pairwise_logrank(self.df, 'uf', 'time', 'event')

        # SP (baixo) vs BA (alto) — ambos bem amostrados
        pair = result[
            ((result['level_a'] == 'SP') & (result['level_b'] == 'BA')) |
            ((result['level_a'] == 'BA') & (result['level_b'] == 'SP'))
        ]
        self.assertFalse(pair.empty)
        self.assertTrue(
            (pair['pvalue'] < 0.05).all(),
            msg=f'SP vs BA deveria ter p < 0.05: {pair[["pvalue"]].values}'
        )

    def test_low_power_flag_for_small_uf(self):
        """
        AM tem apenas 40 contratos → pares envolvendo AM devem ser
        flagged como low_power.
        """
        result = pairwise_logrank(self.df, 'uf', 'time', 'event',
                                  min_events=10)
        am_pairs = result[
            (result['level_a'] == 'AM') | (result['level_b'] == 'AM')
        ]
        self.assertFalse(am_pairs.empty)
        self.assertTrue(
            am_pairs['low_power'].all(),
            msg='AM deveria ter low_power=True em todos os pares'
        )

    def test_no_low_power_for_large_ufs(self):
        """
        SP (n=800) e MG (n=500) têm muitos eventos →
        par SP×MG não deve ser low_power.
        """
        result = pairwise_logrank(self.df, 'uf', 'time', 'event')
        pair = result[
            ((result['level_a'] == 'SP') & (result['level_b'] == 'MG')) |
            ((result['level_a'] == 'MG') & (result['level_b'] == 'SP'))
        ]
        self.assertFalse(pair.empty)
        self.assertFalse(
            pair['low_power'].values[0],
            msg='SP × MG não deve ser low_power'
        )

    def test_output_columns(self):
        """Saída deve ter todas as colunas esperadas."""
        result = pairwise_logrank(self.df, 'uf', 'time', 'event')
        expected_cols = ['level_a','level_b','n_a','n_b',
                         'events_a','events_b','logrank_stat',
                         'pvalue','similar','low_power','interpretation']
        for col in expected_cols:
            self.assertIn(col, result.columns)

    def test_similar_flag_consistent_with_pvalue(self):
        """Flag 'similar' deve ser consistente com pvalue > 0.05."""
        result = pairwise_logrank(self.df, 'uf', 'time', 'event')
        valid  = result.dropna(subset=['pvalue'])
        expected_similar = valid['pvalue'] > 0.05
        pd.testing.assert_series_equal(
            valid['similar'].reset_index(drop=True),
            expected_similar.reset_index(drop=True),
            check_names=False
        )


class TestSuggestCollapsing(unittest.TestCase):
    """Testes para suggest_collapsing."""

    @classmethod
    def setUpClass(cls):
        df       = make_uf_dataset(TRUE_GROUPS, UF_SIZES, random_state=42)
        cls.pair = pairwise_logrank(df, 'uf', 'time', 'event')

    def test_returns_list_of_lists(self):
        """Saída deve ser lista de listas."""
        result = suggest_collapsing(self.pair)
        self.assertIsInstance(result, list)
        for group in result:
            self.assertIsInstance(group, list)

    def test_all_levels_covered(self):
        """Todos os níveis devem aparecer em exatamente um grupo."""
        result  = suggest_collapsing(self.pair)
        all_in  = [lv for group in result for lv in group]
        all_levels = set(self.pair['level_a']) | set(self.pair['level_b'])
        self.assertEqual(set(all_in), all_levels)
        # Sem duplicatas
        self.assertEqual(len(all_in), len(set(all_in)))

    def test_n_groups_matches_true_structure(self):
        """
        Com os dados artificiais, deve sugerir exatamente 3 grupos
        (baixo, médio, alto risco) — excluindo AM por low_power.
        """
        result = suggest_collapsing(self.pair, ignore_low_power=False)
        # AM fica sozinho (low_power → não é colapsado com segurança)
        # então esperamos 3 grupos genuínos + AM isolado = 4 grupos
        self.assertGreaterEqual(len(result), 3)

    def test_baixo_risco_same_group(self):
        """
        SP, MG, RS (baixo risco) devem ser sugeridos no mesmo grupo.
        """
        result = suggest_collapsing(self.pair)
        found = False
        for group in result:
            if 'SP' in group and 'MG' in group and 'RS' in group:
                found = True
                break
        self.assertTrue(
            found,
            msg=f'SP, MG, RS deveriam estar no mesmo grupo. Grupos: {result}'
        )

    def test_sp_and_ba_different_groups(self):
        """
        SP (baixo risco) e BA (alto risco) devem estar em grupos diferentes.
        """
        result = suggest_collapsing(self.pair)
        for group in result:
            self.assertFalse(
                ('SP' in group and 'BA' in group),
                msg='SP e BA não deveriam estar no mesmo grupo'
            )


class TestCollapsingReport(unittest.TestCase):
    """Testes de integração para collapsing_report."""

    @classmethod
    def setUpClass(cls):
        cls.df = make_uf_dataset(TRUE_GROUPS, UF_SIZES, random_state=42)

    def test_output_keys(self):
        """Saída deve ter as chaves esperadas."""
        result = collapsing_report(
            self.df, 'uf', 'time', 'event', verbose=False
        )
        for key in ['pairwise', 'suggested_groups', 'collapse_map']:
            self.assertIn(key, result)

    def test_collapse_map_covers_all_levels(self):
        """collapse_map deve mapear todos os níveis."""
        result = collapsing_report(
            self.df, 'uf', 'time', 'event', verbose=False
        )
        all_levels = set(self.df['uf'].unique())
        self.assertEqual(set(result['collapse_map'].keys()), all_levels)

    def test_collapse_map_groups_baixo_risco(self):
        """
        SP, MG, RS devem ter o mesmo valor no collapse_map
        (mesmo rótulo de grupo).
        """
        result = collapsing_report(
            self.df, 'uf', 'time', 'event', verbose=False
        )
        cmap = result['collapse_map']
        self.assertEqual(
            cmap['SP'], cmap['MG'],
            msg=f'SP e MG deveriam ter o mesmo grupo. SP={cmap["SP"]}, MG={cmap["MG"]}'
        )
        self.assertEqual(
            cmap['MG'], cmap['RS'],
            msg=f'MG e RS deveriam ter o mesmo grupo. MG={cmap["MG"]}, RS={cmap["RS"]}'
        )

    def test_collapse_map_separates_extremes(self):
        """
        SP (baixo) e BA (alto) devem ter grupos diferentes no collapse_map.
        """
        result = collapsing_report(
            self.df, 'uf', 'time', 'event', verbose=False
        )
        cmap = result['collapse_map']
        self.assertNotEqual(
            cmap['SP'], cmap['BA'],
            msg='SP e BA não deveriam estar no mesmo grupo'
        )


# ── Runner ────────────────────────────────────────────────────────────────────
def run_tests(verbosity: int = 2):
    suite = unittest.TestSuite()
    for cls in [
        TestPairwiseLogrank,
        TestSuggestCollapsing,
        TestCollapsingReport,
    ]:
        suite.addTests(unittest.TestLoader().loadTestsFromTestCase(cls))

    buf    = io.StringIO()
    runner = unittest.TextTestRunner(stream=buf, verbosity=verbosity)
    result = runner.run(suite)
    print(buf.getvalue())

    total  = result.testsRun
    failed = len(result.failures) + len(result.errors)
    print('═' * 55)
    print(f'  Total: {total} | ✅ Passou: {total-failed} | ❌ Falhou: {failed}')
    print('═' * 55)
    return result


run_tests()